In [69]:
import numpy as np
import pandas as pd
import itertools

# 演習問題8.2

In [2]:
index = [
    ("Low", "Low"),
    ("Low", "High"),
    ("Medium", "Low"),
    ("Medium", "High"),
    ("High", "Low"),
    ("High", "High")
]

mi = pd.MultiIndex.from_tuples(index, names=["Satisfaction", "Contact"])
df_satisfaction = pd.DataFrame(
    [
        [65, 34, 54, 47, 100, 100],
        [130, 141, 76, 116, 111, 191],
        [67, 130, 48, 105, 62, 104]
    ],
    columns = mi,
    index = ["Tower_block", "Apartment", "House"],
)

In [3]:
df_satisfaction

Satisfaction  Low      Medium      High     
Contact       Low High    Low High  Low High
Tower_block    65   34     54   47  100  100
Apartment     130  141     76  116  111  191
House          67  130     48  105   62  104

## (a)

In [4]:
# 満足度と交流の程度の関係
df_pivot = df_satisfaction.sum().to_frame().reset_index().pivot(index = "Contact", columns = "Satisfaction", values = 0)
df_pivot = df_pivot.reindex(columns = ["Low", "Medium", "High"], index = ["Low", "High"])
df_pivot

Satisfaction,Low,Medium,High
Contact,,,
Low,262,178,273
High,305,268,395


In [5]:
df_pivot / df_pivot.sum().sum()

Satisfaction,Low,Medium,High
Contact,,,
Low,0.15586,0.105889,0.162403
High,0.18144,0.159429,0.234979


In [6]:
df_pivot.div(df_pivot.sum(axis = 1), axis = 0)*100

Satisfaction,Low,Medium,High
Contact,,,
Low,36.746143,24.964937,38.288920
High,31.508264,27.685950,40.805785


In [7]:
# 満足度と住宅の程度の関係
df_pivot = pd.concat([df_satisfaction[s].sum(axis = 1).to_frame() for s in ["Low", "Medium", "High"]], axis = 1)
df_pivot.columns = ["Low", "Medium", "High"]
df_pivot

,Low,Medium,High
Tower_block,99,101,200
Apartment,271,192,302
House,197,153,166


In [8]:
df_pivot / df_pivot.sum().sum()

,Low,Medium,High
Tower_block,0.058894,0.060083,0.118977
Apartment,0.161214,0.114218,0.179655
House,0.117192,0.091017,0.098751


In [9]:
df_pivot.div(df_pivot.sum(axis = 1), axis = 0)

,Low,Medium,High
Tower_block,0.247500,0.252500,0.500000
Apartment,0.354248,0.250980,0.394771
House,0.381783,0.296512,0.321705


In [10]:
# 交流と住宅の程度の関係


In [11]:
df_satisfaction.columns = df_satisfaction.columns.swaplevel(0, 1)
df_pivot = pd.concat([df_satisfaction[s].sum(axis = 1).to_frame() for s in ["Low", "High"]], axis = 1)
df_pivot.columns = ["Low", "High"]
df_pivot

,Low,High
Tower_block,219,181
Apartment,317,448
House,177,339


In [12]:
df_pivot.div(df_pivot.sum(axis = 1), axis = 0)

,Low,High
Tower_block,0.547500,0.452500
Apartment,0.414379,0.585621
House,0.343023,0.656977


## (b)

In [13]:
index = [
    ("Low", "Low"),
    ("Low", "High"),
    ("Medium", "Low"),
    ("Medium", "High"),
    ("High", "Low"),
    ("High", "High")
]

mi = pd.MultiIndex.from_tuples(index, names=["Satisfaction", "Contact"])
df_satisfaction = pd.DataFrame(
    [
        [65, 34, 54, 47, 100, 100],
        [130, 141, 76, 116, 111, 191],
        [67, 130, 48, 105, 62, 104]
    ],
    columns = mi,
    index = ["Tower_block", "Apartment", "House"],
)
df_satisfaction

Satisfaction  Low      Medium      High     
Contact       Low High    Low High  Low High
Tower_block    65   34     54   47  100  100
Apartment     130  141     76  116  111  191
House          67  130     48  105   62  104

In [14]:
# MultiIndex列を縦持ちに変換
df_long = (
    df_satisfaction
    .stack(level=["Satisfaction", "Contact"])
    .reset_index()
    .rename(columns={
        "level_0": "House",
        0: "num_people"
    })
)

/var/folders/g4/gx0098gn4072x6j58_7tw_8c0000gn/T/ipykernel_7731/10023825.py:4: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  .stack(level=["Satisfaction", "Contact"])


In [15]:
df_long

,House,Satisfaction,Contact,num_people
0,Tower_block,High,Low,100
1,Tower_block,High,High,100
2,Tower_block,Low,Low,65
3,Tower_block,Low,High,34
4,Tower_block,Medium,Low,54
5,Tower_block,Medium,High,47
6,Apartment,High,Low,111
7,Apartment,High,High,191
8,Apartment,Low,Low,130
9,Apartment,Low,High,141


In [56]:
# # 目的変数
# y = df_long["Satisfaction"]
# y = pd.get_dummies(y.to_frame(), dtype = int).drop(["Satisfaction_Low"], axis = 1).values

results = {}
configs = {
    "X1": {"cat_cols": ["House"], "drop": ["House_Apartment", "Satisfaction_Low"], "interaction": False},
    "X2": {"cat_cols": ["Contact"], "drop": ["Contact_Low", "Satisfaction_Low"], "interaction": False},
    "X3": {"cat_cols": ["House", "Contact"], "drop": ["House_Apartment", "Contact_Low", "Satisfaction_Low"], "interaction": False},
    "X4": {"cat_cols": ["House", "Contact"], "drop": ["House_Apartment", "Contact_Low", "Satisfaction_Low"], "interaction": True}
}

for key, cfg in configs.items():

    X = pd.get_dummies(
        df_long[cfg["cat_cols"] + ["Satisfaction", "num_people"]], dtype=int, columns=cfg["cat_cols"] + ["Satisfaction"]
    ).drop(cfg["drop"], axis=1)

    # 交互作用（X4 のみ）
    if cfg["interaction"]:
        X["House_House:Contact_High"] = X["House_House"] & X["Contact_High"]
        X["House_Tower_block:Contact_High"] = X["House_Tower_block"] & X["Contact_High"]
        
    #集計
    list_expvar = [v for v in X.columns if (v != "num_people") & ("Satisfaction" not in v)]
    a = X.groupby(list_expvar, as_index = False).sum()[list_expvar+ ["num_people"]]
    b = X[X["Satisfaction_High"] == 1].groupby(list_expvar, as_index = False).sum()[list_expvar + ["num_people"]].rename(columns = {"num_people":"Satisfaction_High"})
    c = X[X["Satisfaction_Medium"] == 1].groupby(list_expvar, as_index = False).sum()[list_expvar + ["num_people"]].rename(columns = {"num_people":"Satisfaction_Medium"})
    df_tmp = pd.merge(pd.merge(a, b, on = list_expvar, how = "left"), c, on = list_expvar, how = "left")

    X = df_tmp[list_expvar].values
    n = df_tmp["num_people"].values
    y = df_tmp[["Satisfaction_Medium", "Satisfaction_High"]].values
    X = np.column_stack([np.ones(len(X)), X])
    results[key] = {"X": X, "y":y,"n": n, "coef": ["intercept"] + list_expvar, "class_label": ["Satisfaction_Medium","Satisfaction_High"]}

In [57]:
results

{'X1': {'X': array([[1., 0., 0.],
         [1., 0., 1.],
         [1., 1., 0.]]),
  'y': array([[192, 302],
         [101, 200],
         [153, 166]]),
  'n': array([765, 400, 516]),
  'coef': ['intercept', 'House_House', 'House_Tower_block'],
  'class_label': ['Satisfaction_Medium', 'Satisfaction_High']},
 'X2': {'X': array([[1., 0.],
         [1., 1.]]),
  'y': array([[178, 273],
         [268, 395]]),
  'n': array([713, 968]),
  'coef': ['intercept', 'Contact_High'],
  'class_label': ['Satisfaction_Medium', 'Satisfaction_High']},
 'X3': {'X': array([[1., 0., 0., 0.],
         [1., 0., 0., 1.],
         [1., 0., 1., 0.],
         [1., 0., 1., 1.],
         [1., 1., 0., 0.],
         [1., 1., 0., 1.]]),
  'y': array([[ 76, 111],
         [116, 191],
         [ 54, 100],
         [ 47, 100],
         [ 48,  62],
         [105, 104]]),
  'n': array([317, 448, 219, 181, 177, 339]),
  'coef': ['intercept', 'House_House', 'House_Tower_block', 'Contact_High'],
  'class_label': ['Satisfactio

In [58]:
y

array([[ 76, 111],
       [116, 191],
       [ 54, 100],
       [ 47, 100],
       [ 48,  62],
       [105, 104]])

In [59]:
n

array([317, 448, 219, 181, 177, 339])

### ロジスティック回帰

In [60]:
from mlogit import MultinomialLogitOptimizer
import numpy as np

In [61]:
results["X1"]["X"].shape

(3, 3)

In [62]:
pd.DataFrame(y).drop_duplicates()

,0,1
0,76,111
1,116,191
2,54,100
3,47,100
4,48,62
5,105,104


In [63]:
results["X1"]["n"].shape

(3,)

In [79]:
cls = MultinomialLogitOptimizer(results["X1"]["X"], results["X1"]["y"], results["X1"]["n"])
cls.fit()
beta, std_error = cls.beta, cls.std_error
print(" ")
pd.DataFrame(
        [beta.flatten() , std_error.flatten()],
        columns = [f"{b}_{a}" for (a, b) in itertools.product(results["X1"]["class_label"], results["X1"]["coef"])],
        index = ["推定値", "標準誤差"]
        ).T

,推定値,標準誤差
intercept_Satisfaction_Medium,-0.344623,0.094331
House_House_Satisfaction_Medium,0.091858,0.143215
House_Tower_block_Satisfaction_Medium,0.364624,0.170001
intercept_Satisfaction_High,0.108308,0.083674
House_House_Satisfaction_High,-0.279524,0.134542
House_Tower_block_Satisfaction_High,0.594889,0.148668


In [80]:
cls.calc_deviance()

TypeError: loop of ufunc does not support argument 0 of type int which has no callable log method

In [27]:
std_error

array([0.09433117, 0.14321473, 0.17000109, 0.08367374, 0.13454194,
       0.14866844])

In [24]:
import itertools
itertools.product( )